   # Ingestion v1: local MySQL to S3 raw (retired)

   First version of the ingestion. Runs on my laptop: reads the next month from
   local MySQL, uploads it to S3 raw as CSV, and records the month in a local
   tracker file. 48 months uploaded this way.

   Replaced by the Glue job `loans_extract_job.py`, which reads from RDS and
   writes Parquet. The raw CSV files from this version were later converted to Parquet.

In [ ]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine, text
from io import StringIO
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

In [ ]:
# CONFIG

S3_BUCKET = "lending-data-de-project" 
S3_PREFIX = "lending-data-monthly/raw/" 
TRACKER_FILE = Path("s3_ingest_tracker.txt")
TABLE_NAME = "loans"

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}


In [ ]:
def get_engine():
    """Create a connection to MySQL using details from .env."""
    user = os.getenv("MYSQL_USER")
    password = os.getenv("MYSQL_PASSWORD")
    host = os.getenv("MYSQL_HOST", "127.0.0.1")
    db = os.getenv("MYSQL_DB")
    return create_engine(f"mysql+pymysql://{user}:{password}@{host}:3306/{db}")

def check_connection(engine):
    """Fail fast if MySQL can't be reached, before the slow CSV load."""
    with engine.connect() as conn:
        db = conn.execute(text("SELECT DATABASE()")).scalar()
    print(f"[check_connection] Connected to database: {db}")


In [ ]:
engine = get_engine()
check_connection(engine)

In [ ]:
def upload_to_s3(df, bucket, key):
    """Turn a DataFrame into CSV text in memory and upload it to S3."""
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"[upload_to_s3] Uploaded {len(df):,} rows to s3://{bucket}/{key}")

In [ ]:
def read_tracker():
    """Return the last month uploaded to S3, e.g. '2015-03'."""
    last_month = TRACKER_FILE.read_text().strip()
    print(f"[read_tracker] Last month uploaded: {last_month}")
    return last_month

def update_tracker(month):
    """Save `month` as the last month uploaded."""
    TRACKER_FILE.write_text(month)
    print(f"[update_tracker] Tracker now says {month}")

def get_next_month(last_month):
    """'2015-03' -> '2015-04' and '2015-12' -> '2016-01'."""
    last_date = pd.to_datetime(last_month, format="%Y-%m")
    next_date = last_date + pd.DateOffset(months=1)
    return next_date.strftime("%Y-%m")

def to_source_format(month):
    """'2015-01' -> 'Jan-2015', the way issue_d is stored in MySQL."""
    month_date = pd.to_datetime(month, format="%Y-%m")
    return month_date.strftime("%b-%Y")


In [ ]:
def run_ingestion():
    """Upload the next month from MySQL to S3. Return True if uploaded, False if nothing left."""

    # 1. Which month is next?
    last_month = read_tracker()
    next_month = get_next_month(last_month)
    source_value = to_source_format(next_month)
    print(f"[run_ingestion] Next month: {next_month} (issue_d = '{source_value}')")

    # 2. Pull that month from MySQL
    query = text(f"SELECT * FROM {TABLE_NAME} WHERE issue_d = :month")
    df = pd.read_sql(query, engine, params={"month":source_value})
    print(f"[run_ingestion] Rows from MySQL: {df.shape}")

    # 3. Nothing left? Stop without touching the tracker
    if df.empty:
        print(f"[run_ingestion] No data for {next_month}. All months uploaded.")
        return False

    # 4. Upload to S3
    s3_key = f"{S3_PREFIX}loans_{next_month}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # 5. Only now, move the tracker on
    update_tracker(next_month)
    return True

In [ ]:
run_ingestion()